# 004: Regularization Strategies — L1, L2, and Elastic Net penalties

## 1. L1 REGULARIZATION (LASSO)
- Adds an absolute value penalty to the weights:
  $$\mathcal{L} = \mathcal{L}_0 + \lambda_1 \sum |w_i|$$
- Drives some weights exactly to zero, performing implicit feature selection (producing **sparse models**).

## 2. L2 REGULARIZATION (RIDGE / WEIGHT DECAY)
- Adds a squared magnitude penalty to the weights:
  $$\mathcal{L} = \mathcal{L}_0 + \frac{\lambda_2}{2} \sum w_i^2$$
- Limits weight magnitudes to prevent overfitting while keeping all parameters active.

## 3. ELASTIC NET
- Combines both L1 and L2 penalties:
  $$\mathcal{L} = \mathcal{L}_0 + \lambda_1 \sum |w_i| + \frac{\lambda_2}{2} \sum w_i^2$$
---


In [ ]:
import numpy as np

class RegularizedLinearRegression:
    """Linear regression model with support for L1, L2, and Elastic Net regularization."""
    def __init__(self, input_dim):
        self.w = np.random.randn(1, input_dim) * 0.1
        self.b = 0.0

    def forward(self, X):
        return self.w @ X + self.b

    def compute_loss(self, y_true, y_pred, l1_lambda=0.0, l2_lambda=0.0):
        m = y_true.shape[1]
        base_loss = (1 / (2 * m)) * np.sum((y_true - y_pred) ** 2)
        
        # Add L1 and L2 penalties
        l1_penalty = l1_lambda * np.sum(np.abs(self.w))
        l2_penalty = (l2_lambda / 2.0) * np.sum(self.w ** 2)
        
        return base_loss + l1_penalty + l2_penalty

    def backward(self, X, y_true, y_pred, l1_lambda=0.0, l2_lambda=0.0):
        m = X.shape[1]
        error = y_pred - y_true
        
        # Base gradients
        dw = (1 / m) * error @ X.T
        db = (1 / m) * np.sum(error)
        
        # Add regularization gradients
        dw += l1_lambda * np.sign(self.w)  # Subgradient for L1
        dw += l2_lambda * self.w          # Gradient for L2
        
        return dw, db



In [ ]:
# Testing the regularization shrinkage
print("--- Regularization Shrinkage Demonstration ---")
np.random.seed(42)
X = np.random.randn(5, 50)
y = 3.0 * X[0:1] + np.random.randn(1, 50) * 0.1  # Only first feature is important

# 1. Train L1 model (Lasso)
model_l1 = RegularizedLinearRegression(input_dim=5)
lr = 0.05
for _ in range(500):
    y_p = model_l1.forward(X)
    dw, db = model_l1.backward(X, y, y_p, l1_lambda=0.5, l2_lambda=0.0)
    model_l1.w -= lr * dw
print(f"L1 Weights: {model_l1.w.flatten().round(4)}") # Features 1-4 should shrink to near 0

# 2. Train L2 model (Ridge)
model_l2 = RegularizedLinearRegression(input_dim=5)
for _ in range(500):
    y_p = model_l2.forward(X)
    dw, db = model_l2.backward(X, y, y_p, l1_lambda=0.0, l2_lambda=0.5)
    model_l2.w -= lr * dw
print(f"L2 Weights: {model_l2.w.flatten().round(4)}") # Weights are small but non-zero
